# SmolBench on Amazon Bedrock: a three-model ladder trio

This notebook runs both SmolBench studies on Amazon Bedrock, with reasoning switched on for
every model:

- **Periodic induction**: the paper arms `intens`, `noise_intens` and `extens` plus the
  review arm `related_intens`, seeds 0–29, 9 count questions per replicate.
  `related_intens` is the rules followed by irrelevant rules of the same form (a period of
  the replicate with a fresh label, which no question asks about), filled to the token
  count of `extens`: padding that is irrelevant but related, where `noise_intens` pads with
  unrelated whitespace.
- **Horn-rule deduction**: the four paper arms `lem`, `pad`, `disc` and `both` plus the four
  review arms `fwd`, `tjunk`, `rep` and `marked-lemma`, seeds 100–199, 3 replicates,
  temperature 0.7, each model at its ICLR chain length `m`.

The three models are all in the ICLR ladder, come from three different labs, and are
served by Bedrock:

| roster key | Bedrock id | lab | parameters | Horn `m` |
|---|---|---|---|---|
| `gemma-4-31b` | `google.gemma-4-31b` (served under `/openai/v1`) | Google | 30.7B dense | 64 |
| `nemotron-3-super-120b-a12b` | `nvidia.nemotron-super-3-120b` | NVIDIA | 120B MoE, 12B active | 20 |
| `deepseek-v3.1` | resolved from `GET /models` | DeepSeek | 685B MoE, 37B active | 64 |

The steps between them are 3.9x and 5.7x, or 22x end to end, in total parameters. Bedrock
hosts no other three ladder models from different large labs. Qwen3.5 is only on
SageMaker, and Gemma-4 E2B/12B and Nemotron-3 Nano-4B are not hosted.

**Code reused.** Requests go through `smolbench.evals.providers.aws`, the OpenAI-compatible
bedrock-mantle client. The induction quizzes come from
`notebooks/induction/run_study.make_quizzes`, and the runs use `ReplicateHarness`. The
Horn rungs are rendered by `smolbench.deduction.horn.cli.write_seed` (the renderer behind
`horn.repro.render_rung`, here with all eight arms) and the paper arms are checked against
the ICLR digests; each cell is run and scored by `scripts/deduction/horn/sweep.run_cell`.
`bedrock_trio.py` adds only the roster, a per-model endpoint switch, the results store, and
the reasoning checks.

**Differences from the ICLR runs:**

- **Output budgets.** They are the ICLR ones. Horn sends a 131,072-token cap, cut per cell
  so that prompt and output fit a 131,072-token context (`sampling` in `horn/iclr.json`).
  Induction sends `run_study.completion_budget`: that context less the longest prompt and
  a template reserve. bedrock-mantle honours both; `cap_probe.py` returned completions of
  up to 131,072 tokens on 2026-11-13. The Bedrock model cards list output caps of 8K
  (DeepSeek V3.1) and 32K (Nemotron Super). The first smoke sent those and most of its
  Horn answers were truncated; its rows are under `bedrock-trio/smoke/` and are not read.
  A truncated answer scores `length` (Horn) or `null`, counted as wrong (induction). Both
  rates are reported below.
- **Induction temperature.** It is not sent, as at ICLR, but the server default here is
  Bedrock's, not the vLLM `generation_config.json` default the ICLR runs used.
- **Seeding.** Every request here is seeded: the replicate seed for induction and the
  replicate index for Horn. The published DeepSeek and Nemotron Horn rows came from
  unseeded Bedrock Converse calls; the Gemma rows and all published induction rows came
  from self-hosted vLLM.

**Running it.**
1. Refresh your AWS credentials.
2. Set `SMOKE` in the next cell. `True` runs a few seeds per model; `False` runs the full
   protocol.
3. Run all cells.

**Where results go.** The private bucket `RESULTS_S3` is the store of record; objects there
are append-only. `rebuttal/results/` is a local mirror of it that the report cells read; it
is not committed. A rerun resumes from the bucket, on this machine or any other: finished
replicates and rows are skipped. A smoke run uses the `smoke-2/` sub-prefix and
`rebuttal/results/smoke-2/`, so it never mixes with protocol runs. Rendered rungs go to `rebuttal/rungs/` (not committed,
since they are deterministic and digest-checked).

**The `extens` arm.** On this branch the `extens` context is the intensional rules followed
by the listing. The published `extens` and `noise_intens` numbers come from the earlier
`extens` arm, which carried the listing only, so only `intens` is like-for-like with the
published values.

In [ ]:
import json
import os
import sys
from pathlib import Path

# The fleet's weight mirror, read by the EC2 provider when it is imported. Only the
# attention replay uses it; an EC2 instance without it fails to start the model.
os.environ.setdefault("EC2_S3_MODEL_CACHE", "s3://smolbench-model-cache-414266451290/hf")
os.environ.setdefault("EC2_S3_CACHE_REGION", "us-west-2")

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "rebuttal" / "bedrock_trio.py").exists())
sys.path.insert(0, str(REPO_ROOT / "rebuttal"))

import bedrock_trio as bt
from smolbench.deduction.horn import repro as horn_repro
from smolbench.evals.results_store import format_run_ts, utcnow
from smolbench.induction import repro as induction_repro

SMOKE = True
REGION = "us-east-1"  # bedrock-mantle is verified here
# The store of record; None keeps results local only. A smoke run gets its own prefix and
# mirror, because the bucket is append-only and the protocol prefix must stay clean.
RESULTS_S3 = bt.RESULTS_S3 + (f"/{bt.SMOKE_PREFIX}" if SMOKE else "")
RESULTS = bt.RESULTS_DIR / bt.SMOKE_PREFIX if SMOKE else bt.RESULTS_DIR  # local mirror
HORN_ARMS = bt.HORN_ARMS  # the four paper arms, then the four review arms
MODELS = list(bt.TRIO)  # roster keys, smallest first
CONCURRENCY = 8  # parallel requests per model

if SMOKE:
    INDUCTION_SEEDS, HORN_SEEDS, HORN_REPS = [0, 1], [100, 101], 1
else:
    INDUCTION_SEEDS, HORN_SEEDS, HORN_REPS = list(range(30)), list(range(100, 200)), 3

print(f"SMOKE={SMOKE}: induction seeds {INDUCTION_SEEDS[0]}-{INDUCTION_SEEDS[-1]}, "
      f"Horn seeds {HORN_SEEDS[0]}-{HORN_SEEDS[-1]} x {HORN_REPS} reps, "
      f"results in {RESULTS_S3 or RESULTS}")

## Authenticate and resolve the model ids

This mints a short-term Bedrock bearer token from your AWS profile. An
`AWS_BEARER_TOKEN_BEDROCK` already set in the environment is used as is. When `RESULTS_S3`
is set, it also checks that the results bucket can be listed, so a credential problem fails
here and not after inference has been billed. The listing combines both mantle paths,
because Gemma 4 is only served under `/openai/v1`.

In [ ]:
bt.mint_token(REGION)
if RESULTS_S3:
    bt.check_bucket(RESULTS_S3)
listing = set()
for path in ("/v1", "/openai/v1"):
    try:
        listing |= set(bt.list_mantle_models(REGION, path))
    except Exception as err:  # one path failing is reported, not fatal
        print(f"GET {path}/models failed: {err}")
TRIO = bt.resolve_ids(sorted(listing), {k: bt.TRIO[k] for k in MODELS})
for m in TRIO.values():
    print(f"{m.key:28s} {m.bedrock_id:34s} {bt.base_url(m, REGION)}")

## Force reasoning

For each model, this sends a one-line question with each of the model's reasoning fields
in turn and keeps the first one that returns reasoning. If none does, the cell fails, and
the model is not run without reasoning. The probes are saved to `results/probes.json` and,
when `RESULTS_S3` is set, uploaded to the bucket as `probes/probes--<stamp>.json`.

In [ ]:
REASONING, PROBES = {}, {}
for key, model in TRIO.items():
    REASONING[key], PROBES[key] = bt.pick_reasoning_args(model, REGION)
    print(f"{key:28s} {REASONING[key]}  <- {PROBES[key][-1]}")
RESULTS.mkdir(parents=True, exist_ok=True)
probes_path = RESULTS / "probes.json"
probes_path.write_text(json.dumps(
    {"region": REGION, "ids": {k: m.bedrock_id for k, m in TRIO.items()},
     "reasoning_args": REASONING, "probes": PROBES}, indent=2) + "\n")
if RESULTS_S3:
    probe_key = bt.push_file(probes_path, RESULTS_S3, f"probes/probes--{format_run_ts(utcnow())}.json")
    print(f"probe record uploaded to {probe_key}")

## Check the induction prompts against the released ones

For seed 0 of each model, the regenerated prompts are compared with the prompts in the
released ICLR results, which are read anonymously from the public bucket. The tokenizers
come from Hugging Face at the pinned revisions, so the gated `google/gemma-4-31B-it` needs
`HF_TOKEN` unless it is already cached. What is checked differs by arm:

- **`intens`**: each prompt must equal the released prompt byte for byte.
- **`extens`**: on this branch the context is the intensional rules followed by the
  listing, while the released prompts carry the listing only. Each prompt, with the rules
  block removed from directly before the listing, must equal the released prompt byte for
  byte. The rules block is taken from the question's own `intens` prompt, so a rules block
  that differs from it fails.
- **`noise_intens`**: it is padded to the length of `extens`, which is longer here than in
  the release, so it cannot equal the released prompt. Its token count under the model's
  own tokenizer must equal that of the matching `extens` prompt.
- **`related_intens`**: it has no released counterpart, so it is checked against its own
  definition: the token count equals that of `extens`; the prompt starts with the `intens`
  prompt up to the end of the rules; the question is identical; every added line is a rule
  with one of the replicate's periods and a fresh label that neither contains nor sits
  inside a real label; no (period, label) pair repeats; and all questions carry prefixes of
  one stream. A few whitespace tokens after the last added line make the count exact.

The published `extens` and `noise_intens` numbers come from the earlier listing-only
`extens` arm, so only `intens` is like-for-like with the published values.

In [ ]:
run_study = bt.load_run_study()
for key in TRIO:
    related = bt.check_induction_prompts(bt.induction_quizzes(0, key), bt.released_queries(key, 0), run_study.for_model(key))
    added = sorted({(r["added_lines"], r["pad_tokens"]) for r in related})
    print(f"{key}: prompts consistent with the released ICLR prompts; "
          f"related_intens (added rule lines, whitespace tokens): {added}")

## Run the induction study

The four arms of one seed are sent together, and the run resumes per arm: a bucket that
already holds a replicate's three paper arms gets only the 9 `related_intens` requests. One YAML per replicate is written to the
bucket as `induction/<model>/seed=<seed>/<arm>--<timestamp>.yaml` below the prefix, the
released-results layout; with `RESULTS_S3 = None` it is written to `results/` instead. When
the run ends, the bucket's replicates are mirrored to
`results/induction/induction/<model>/seed=<seed>/`, which is where the report reads them.

In [ ]:
harness = bt.induction_harness(list(TRIO.values()), INDUCTION_SEEDS, results_dir=RESULTS, s3_uri=RESULTS_S3)
for key, model in TRIO.items():
    print(f"== {key}")
    bt.run_induction(harness, model, REGION, REASONING[key], max_parallel=CONCURRENCY)
if RESULTS_S3:
    print(f"mirrored {bt.pull_induction(RESULTS_S3, RESULTS)} files to {RESULTS / 'induction'}")

## Run the Horn study

Each model's rung is rendered with all eight arms and its paper arms are checked against the
ICLR digests, then every (arm, seed, replicate) cell is sent and scored with the ICLR answer
extraction. Rows are appended to `results/horn/<model>.jsonl` and, when `RESULTS_S3` is set,
pushed to the bucket as `deduction/horn/<model>/rows--<stamp>.jsonl` parts (every 200 new
rows, and when the run ends or is interrupted). Before it picks the cells to run, a model's
run pulls the bucket's rows, so rows collected on any machine are not sent again. A transport
failure is written as an `exception` row and is retried on the next run.

In [ ]:
for key, model in TRIO.items():
    m = bt.horn_m(key)
    rung = bt.render_horn_rung(m, HORN_SEEDS, arms=HORN_ARMS)
    rows = bt.run_horn(model, REGION, REASONING[key], rung, HORN_SEEDS, HORN_REPS,
                       concurrency=CONCURRENCY, out=bt.horn_rows_path(key, RESULTS),
                       arms=HORN_ARMS, s3_uri=RESULTS_S3)
    verdicts = {}
    for r in rows:
        verdicts[r["verdict"]] = verdicts.get(r["verdict"], 0) + 1
    print(f"== {key} m{m}: {len(rows)} new rows {verdicts}")

## Results

When `RESULTS_S3` is set, the first cell below pulls both studies from the bucket into the
local mirror, so a fresh checkout can report from the bucket alone: it needs the config cell
and AWS credentials for the bucket, nothing from Bedrock.

Accuracies and pass rates are shown next to the published ICLR values. Only `intens` is
like-for-like with the published induction values (see the `extens` note above). For Horn,
`horn_repro.report` gives the contrasts (`lem-both`, `pad-both`, `disc-both`) for the four
paper arms only; the four review arms (`fwd`, `tjunk`, `rep`, `marked-lemma`) appear in the
per-arm pass rates and in the per-arm breakdown. The breakdowns show how many answers were
truncated or could not be graded, and how many responses came back without reasoning, which
must be zero. `empty_response` counts induction answers that came back with reasoning text
and no response. Bedrock sometimes returns the bare answer in the reasoning channel and
nothing in the response; such a mark is graded as wrong (`null`), as an empty answer was at
ICLR, and it passes the reasoning check although no reasoning was done.

The Horn reference table reads `notebooks/deduction/review/bedrock_cells.csv`: the paper's
Bedrock cells for `lem`, `pad`, `disc` and `both` (September 2026) and the review-round
cells for `rep` and `tjunk` (November 2026), 300 cells per arm, with the length of their
answers. They ran through the Converse API in us-east-2 with the same budget and without a
seed. Gemma 4 31B ran on vLLM at ICLR and has no row there.

In [ ]:
if RESULTS_S3:
    print(f"induction: {bt.pull_induction(RESULTS_S3, RESULTS)} files written to {RESULTS / 'induction'}")
    for k in MODELS:
        print(f"horn {k}: {bt.pull_horn_rows(k, RESULTS_S3, RESULTS)} rows appended")
else:
    print("RESULTS_S3 is None: reporting from the local results")

In [ ]:
print(induction_repro.report(RESULTS / "induction"))

In [ ]:
rows_files = [bt.horn_rows_path(k, RESULTS) for k in MODELS if bt.horn_rows_path(k, RESULTS).exists()]
print(horn_repro.report(rows_files))

In [ ]:
ref = bt.horn_reference(MODELS)
print("published Bedrock Horn cells (Converse, us-east-2, same output budget)")
print(f"{'model':28s} {'m':>3s} {'arm':6s} {'pass':>6s} {'median tokens':>14s} {'longest':>8s}")
for (k, arm), v in sorted(ref.items(), key=lambda kv: (kv[0][0], bt.HORN_ARMS.index(kv[0][1]))):
    print(f"{k:28s} {v['m']:3d} {arm:6s} {v['pass']:6.1f} {v['tokens_median']:14d} {v['tokens_max']:8d}")

In [ ]:
ind = bt.induction_reasoning_report(RESULTS)
horn = bt.horn_breakdown(rows_files)
print("induction (model, arm): counts")
for k, v in sorted(ind.items()):
    print(f"  {k}: {v}")
print("horn (model, arm): counts")
for k, v in sorted(horn.items()):
    print(f"  {k}: {v}")
unreasoned = {k: v["unreasoned"] for k, v in {**ind, **horn}.items() if v.get("unreasoned")}
assert not unreasoned, f"responses without reasoning: {unreasoned}"

## Attention replay (off by default)

Bedrock returns no attention weights. This section replays the stored transcripts of both
studies (prompt, returned reasoning, answer) through the matching open-weights checkpoint
on a self-hosted vLLM server, teacher-forced, and stores attention summaries under
`<RESULTS_S3>/attention-v2/`. For each response token, layer and head it measures:

- the **prompt entropy**: the Shannon entropy, in nats, of the attention weights on the
  input prompt after they are renormalized to sum to 1;
- the entropy of the whole attention row (prompt and response so far);
- the attention mass on the informative prompt, on the padding (the whitespace of
  `noise_intens`, the added rule lines of `related_intens`), on template tokens and on the
  response so far.

`ATTENTION_MODE = "summary"` stores, per layer and head, the sums of these over the response
tokens, which is all the report needs. `"mass"` stores one value per response token
(induction only; about 35 MB per mark for Gemma 4 31B).

The report averages the prompt entropy over one response (its tokens, layers and heads)
and then over responses, so every response has the same weight. A response token that
cannot attend to any prompt token has no prompt entropy and is left out. That happens in
sliding-window layers once the response is longer than the window, so global-attention
layers and sliding-window layers are also reported apart.

The captures measure the open checkpoint reading Bedrock's text, not Bedrock's own
serving; `attention_replay.APPROXIMATIONS` lists where the two can differ.

- `ATTENTION = True` plans the replay: jobs, token counts, errors and a storage estimate.
  Planning reads the local mirror and the cached tokenizer and chat template. It makes no
  AWS call and costs nothing.
- **A launch needs `ATTENTION_APPROVED = True`.** It provisions one billed EC2 spot GPU
  instance per model and study, serves the checkpoint, captures, audits every stored array
  and terminates the instance. Anything but the literal `True` is refused before any AWS
  client is built.
- A mark whose stored text is mis-decoded is listed as an error and never sent.
- Jobs already in the bucket are skipped, and uploads cannot replace an object.
- `deepseek-v3.1` also needs `ATTENTION_ALLOW_UNVERIFIED = True`. The pinned vLLM image
  serves it only when the weight mirror holds the repository's metadata archive. Marks
  longer than the capture context (`ATTENTION_CAPTURE_CONTEXT`) are excluded.
- The Horn replay takes the arms, seeds and replicates in `ATTENTION_HORN`. The captures of
  2026-11-17 used the arms `lem`, `pad`, `both` and `tjunk`, seeds 100 to 189 and replicates
  0 to 2: 270 responses per arm, as for induction.
- The Gemma pilot of 2026-11-16 is under `<RESULTS_S3>/attention/`. It predates the prompt
  entropy and the `related_intens` arm, and it is not read here.

In [ ]:
ATTENTION = False                  # plan the replay (offline, free) and report stored captures
ATTENTION_APPROVED = False         # launch billed EC2 spot instances; set True only on purpose
ATTENTION_ALLOW_UNVERIFIED = False # also needed for deepseek-v3.1
ATTENTION_MODELS = ("gemma-4-31b",)
ATTENTION_STUDIES = ("induction", "horn")
ATTENTION_MODE = "summary"
ATTENTION_HORN = {"arms": ("lem", "pad", "both", "tjunk"), "seeds": range(100, 190), "reps": (0, 1, 2)}
ATTENTION_CAPTURE_CONTEXT = {"deepseek-v3.1": 122880}  # tokens; 131072 does not start on 8 x H200
ATTENTION_GPU_SHARE = {"deepseek-v3.1": 0.98}          # vLLM GPU memory share that context needs

def attention_kwargs(study, key):
    # The keywords plan_replay and replay_model share for one study and model.
    kwargs = {"study": study, "mode": ATTENTION_MODE, "capture_max_model_len": ATTENTION_CAPTURE_CONTEXT.get(key)}
    if study == "horn":
        kwargs.update(ATTENTION_HORN, horn_rows=bt.horn_rows_path(key, RESULTS), rungs_dir=bt.RUNGS_DIR)
    return kwargs

if ATTENTION:
    import attention_replay as ar

    assert RESULTS_S3, "the attention captures are stored in the bucket: set RESULTS_S3"
    ATTENTION_S3 = f"{RESULTS_S3}/attention-v2"
    for study in ATTENTION_STUDIES:
        for key in ATTENTION_MODELS:
            plan = ar.plan_replay(RESULTS / "induction", [key], s3_uri=ATTENTION_S3, **attention_kwargs(study, key))[key]
            print(plan.summary())
            for err in plan.errors[:10]:
                print("  error:", err)
else:
    print("ATTENTION is False: nothing planned")

In [ ]:
if ATTENTION and ATTENTION_APPROVED is True:
    for study in ATTENTION_STUDIES:
        for key in ATTENTION_MODELS:
            report = ar.replay_model(key, RESULTS / "induction", ATTENTION_S3, approved=ATTENTION_APPROVED,
                                     allow_unverified_serve=ATTENTION_ALLOW_UNVERIFIED,
                                     gpu_memory_utilization=ATTENTION_GPU_SHARE.get(key), **attention_kwargs(study, key))
            print(f"{study} {key}: {len(report.verified)} verified, {len(report.failed)} failed, "
                  f"{len(report.audit_failures)} failed the audit, {len(report.skipped_existing)} already stored, "
                  f"{len(report.partial)} partial, {len(report.manifests)} manifests")
            for job, why in {**report.failed, **report.audit_failures}.items():
                print(f"  {job}: {why}")
            if report.manifest_error:
                print("  manifest error:", report.manifest_error)
else:
    print("attention replay not launched (it needs ATTENTION = True and ATTENTION_APPROVED = True)")

### Attention report

Mirrors the stored captures and prints, per model, study and arm, the mean prompt entropy
(nats) with its standard deviation over responses, the same mean over global-attention
layers and over sliding-window layers, the mean entropy of the whole row, and the pad-mass
ratio (the attention mass on the padding relative to a uniform spread; defined for the
padded induction arms). Only captures that a manifest records as verified are read. For
Gemma 4 31B read the `global` column: in its sliding-window layers only the first response
tokens can reach the prompt. The last column is a 95% bootstrap interval of the mean prompt
entropy (`attention_replay.entropy_ci`): each draw resamples whole seeds, because the
responses of one seed share their context.

In [ ]:
if ATTENTION:
    ATTENTION_MIRROR = RESULTS / "attention-v2"
    mirrored = sum(ar.pull_captures(ATTENTION_S3, ATTENTION_MIRROR, ATTENTION_MODELS, study=study) for study in ATTENTION_STUDIES)
    print(f"{mirrored} capture files mirrored to {ATTENTION_MIRROR}")
    for key in ATTENTION_MODELS:
        layers = "global" if key == "gemma-4-31b" else "all"  # the layers the interval is for
        print(f"{key}\n  {'study':10s} {'arm':15s} {'n':>4s} {'prompt H':>9s} {'sd':>6s} {'global':>7s} {'window':>7s} {'row H':>7s} {'pad ratio':>10s}  95% interval ({layers} layers)")
        for (study, arm), s in ar.read_entropy_table(key, ATTENTION_MIRROR).items():
            _mean, lower, upper, _n, n_seeds = ar.entropy_ci(s, layers=layers)
            print(f"  {study:10s} {arm:15s} {s['prompt_entropy_jobs']:4d} {s['prompt_entropy_mean']:9.3f} {s['prompt_entropy_sd']:6.3f} "
                  f"{s['prompt_entropy_global_mean']:7.3f} {s['prompt_entropy_window_mean']:7.3f} "
                  f"{s['entropy_mean']:7.3f} {s['ratio_mean']:10.4f}  [{lower:.3f}, {upper:.3f}] over {n_seeds} seeds")
else:
    print("ATTENTION is False: no attention report")